# CW_10
# Basics of Digital Holography

In this exercise you will put together a simple digital holography setup to extract the phase of an object of interest. Recall from the course that a holographic measurement can be performed according to the following logic.

Let S be a signal beam that encodes your object given by 
$$
S(x,y)=\|S(x,y)\| e^{j \phi_S (x,y)}
$$

Let R be a reference beam coming in at an angle given by 
$$
R(x,y)=\|R(x,y)\| e^{-j K_x x - j K_y y}
$$

Then if we measure the intensity of their sum, we get
$$
I = \|S+R\|^2 = (S+R)(S+R)^* = \|S\|^2 + \|R\|^2 + SR^* + S^*R
$$
or
$$
I = \|S\|^2 + \|R\|^2 + S\|R\|e^{+ j K_x x + j K_y y} + S^*\|R\| e^{-j K_x x - j K_y y}
$$

In Fourier space, this gives us
$$
\mathcal{F}\{I\} = \mathcal{F}\{\|S\|^2 + \|R\|^2\} + \mathcal{F}\{\|R\|S\}*\delta(k_x-K_x, k_y-K_y) + \mathcal{F}\{\|R\|S^*\}*\delta(k_x+K_x, k_y+K_y)
$$

This representation allows us to isolate the DC and the first orders in space. By masking to isolate just the first order, we can then reconstruct the original object S. 

*Note: This formulation is slightly different than that seen in class, as here we assume that R is flat-phase but not necessarily constant amplitude.*


In [ ]:
# - No modification necessary -

import numpy as np
import matplotlib.pyplot as plt
from skimage import data, transform

# ----------------------------
# Global simulation parameters
# ----------------------------
wavelength = 633e-9  # meters (HeNe laser)
k = 2 * np.pi / wavelength

N = 512              # grid size
dx = 6.5e-6          # pixel pitch (meters)
L = N * dx           # total size

z = 0.02             # propagation distance (meters)

# ----------------------------
# Coordinate grids
# ----------------------------
x = np.linspace(-L/2, L/2, N)
y = np.linspace(-L/2, L/2, N)
X, Y = np.meshgrid(x, y)

# ----------------------------
# Frequency grids
# ----------------------------
fx = np.fft.fftfreq(N, dx)
fy = np.fft.fftfreq(N, dx)
FX, FY = np.meshgrid(fx, fy)

def angular_spectrum_propagation(field, z):
    H = np.exp(1j * k * z * np.sqrt(1 - (wavelength * FX)**2 - (wavelength * FY)**2))
    F = np.fft.fft2(field)
    propagated = np.fft.ifft2(F * H)
    return propagated

def plot_field(field, title="Field"):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    # Intensity
    im0 = axes[0].imshow(np.abs(field)**2, cmap='inferno')
    axes[0].set_title(f"{title} - Intensity")
    plt.colorbar(im0, ax=axes[0])

    # Phase
    im1 = axes[1].imshow(np.angle(field), cmap='twilight')
    axes[1].set_title(f"{title} - Phase")
    plt.colorbar(im1, ax=axes[1])

    plt.tight_layout()
    plt.show()

# Part 1

In this part you will create the source and reference beams for performing your holographic measurement. 

Both of these beams will have the same gaussian amplitude profile. The phase of the reference beam should be a linear ramp corresponding to the desired tilt, and the phase of the source beam should be taken from the provided phase object.

Implement this behavior in the provided cells below and then answer the following questions:
1) What does changing the tilt of the reference beam do for us? Is it better to have high or low angles of incidence? 
2) What is the main factor that limits the range of angles that we can use for our measurement?

In [ ]:
# - No modification necessary -

def load_phase_object(N):
    img = data.page()
    
    # Get original shape
    h, w = img.shape
    
    # Determine padding to make it square
    if h > w:
        pad_total = h - w
        pad_left = pad_total // 2
        pad_right = pad_total - pad_left
        padding = ((0, 0), (pad_left, pad_right))
    else:
        pad_total = w - h
        pad_top = pad_total // 2
        pad_bottom = pad_total - pad_top
        padding = ((pad_top, pad_bottom), (0, 0))
    
    # Pad with zeros (black)
    img_padded = np.pad(img, padding, mode='constant', constant_values=0)
    
    # Resize to NxN
    img_resized = transform.resize(img_padded, (N, N))
    
    # Normalize to phase range [0, π]
    phase = img_resized / np.max(img_resized) * np.pi
    
    return phase


In [ ]:
def gaussian_beam(w0):
    raise NotImplementedError

def generate_signal_beam(w0):
    raise NotImplementedError

def generate_reference_beam(w0, theta_x=0, theta_y=0):
    raise NotImplementedError

In [ ]:
# - No modification necessary

w0 = 1.5e-3
theta_x = 1.5 * np.pi / 180
theta_y = -1.5 * np.pi / 180

signal = generate_signal_beam(w0)
plot_field(signal, "Signal Beam")

signal_propagated = angular_spectrum_propagation(signal, z)
plot_field(signal_propagated, "Propagated Signal Beam")

ref_beam = generate_reference_beam(w0, theta_x, theta_y)
plot_field(ref_beam, "Reference Beam")

## Discussion
TODO

# Part 2

Now that you have created the two beams, you must calculate the detected image and process it to isolate the first diffraction order.

To do this, first combine the two beams and use the square law to digitally detect the field. Then take a Fourier transform (and use an fftshift) to put the image into Fourier space with (0,0) at the center of the image.

After you have the Fourier space representation of the image, you must isolate the 1st diffraction order. To do this, you should first calculate the expected center based on the angle of the reference beam. Then you will need to manually choose a radius that allows you to capture the bulk of the detail in the order without overlapping the 0th order. Implement this functionality in the provided functions first_order_location and circular_crop, and use the provided visualization script to verfiy your implementaiton.

Then, answer the following questions:
1) What is the difference between the two first order spots?
2) How can you tell which one is which?

In [ ]:
detected_intensity = TODO
H_f = TODO

In [ ]:
# - No modification necessary -

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Detected intensity
im0 = axes[0].imshow(detected_intensity, cmap='gray_r')
axes[0].set_title("Detected Intensity")
plt.colorbar(im0, ax=axes[0])

# Fourier spectrum (log scale)
im1 = axes[1].imshow(np.log(np.abs(H_f) + 1), cmap='viridis')
axes[1].set_title("Fourier Spectrum of Detected Intensity")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
def first_order_location(theta_x, theta_y):
    raise NotImplementedError

In [ ]:
radius = TODO

def circular_crop(F, center, radius):
    raise NotImplementedError

In [ ]:
# - No modification necessary -

center = first_order_location(theta_x, theta_y)
H_crop = circular_crop(H_f, center, radius)

plt.figure(figsize=(5,5))
plt.imshow(np.log(np.abs(H_crop)+1), cmap='viridis')
plt.colorbar()
plt.title("Isolated First Order")
plt.show()

## Discussion
TODO

# Part 3
Now that we have isolated the first order, we can finish processing the information to recreate the original object.

First, we will need to remove the reference beam. For simplicity, we will remove it in two parts. First, we will remove the tilt of the beam by returning the first order to the center of the image. Then, after taking the inverse Fourier transform, we will remove the envelope of the beam by dividing out its amplitude. Lastly, we will need to take the reconstructed field, which is that of the object at the detector, and reverse propagate it back to the original location of the object to get the original field.

To do this, implement the functions recenter_spectrum and reconstruct_field.

Then, answer the following questions:
1) In this implementation we chose to translate the first order back to the center and then take the inverse Fourier transform. How could the reconstruction process be done if we wanted to take a Fourier transform first?
2) Is the reconstruction we get perfect? What do the major discrepancies look like and what might be their cause?

In [ ]:
def recenter_spectrum(F, center):
    raise NotImplementedError

In [ ]:
# - No modification necessary -

H_centered = recenter_spectrum(H_crop, center)

plt.figure(figsize=(5,5))
plt.imshow(np.log(np.abs(H_centered)+1), cmap='viridis')
plt.colorbar()
plt.title("Recentered Spectrum")
plt.show()

In [ ]:
def reconstruct_field(F_centered, reference):
    raise NotImplementedError

In [ ]:
# - No modification necessary -

reconstructed = reconstruct_field(H_centered, ref_beam)
recovered_object = angular_spectrum_propagation(reconstructed,-z)

plot_field(reconstructed, "Reconstructed Field")
plot_field(recovered_object, "Recovered Object Field")

## Discussion
TODO